# T5 Zero-Shot PDE Dialect Evaluation
Quick evaluation across four PDE dialects with retry logic, debug logging, and file outputs.

## Cell 1 — Config

In [117]:
CFG = {
    "model_name": "google/flan-t5-large",
    "dialects": ["natural", "prefix", "postfix", "latex"],
    "n_target": 30,
    "max_attempts": 3,
    "global_retry_limit": 100,
    "seed": 42,
    "max_new_tokens": 64,
}

FAMILY_LABELS   = ["Advection", "Burgers", "Heat", "KleinGordon", "Laplace", "Wave"]
OPERATOR_LABELS = ["sin", "cos", "exp", "tanh", "polynomial"]

PROMPT_TEMPLATE = (
    "You are a classifier. Read the PDE expression and output exactly one line with the family and operators.\n"
    "Allowed family labels: Advection, Burgers, Heat, KleinGordon, Laplace, Wave\n"
    "Allowed operator labels: sin, cos, exp, tanh, polynomial\n\n"
    "Example 1:\n"
    "dialect: natural\n"
    "expression: The time derivative of u plus the spatial derivative of u is zero.\n"
    "Output: family=Advection;operators=polynomial\n\n"
    "Example 2:\n"
    "dialect: latex\n"
    "expression: \\partial_t u = \\alpha \\nabla^2 u + \\sin(x)\n"
    "Output: family=Heat;operators=polynomial,sin\n\n"
    "Example 3:\n"
    "dialect: prefix\n"
    "expression: +(*(-1, d(u, t)), *(u, d(u, x)))\n"
    "Output: family=Burgers;operators=polynomial\n\n"
    "Example 4:\n"
    "dialect: latex\n"
    "expression: \\nabla^2 u = \\exp(x)\n"
    "Output: family=Laplace;operators=exp\n\n"
    "Example 5:\n"
    "dialect: natural\n"
    "expression: second derivative in time equals second derivative in space.\n"
    "Output: family=Wave;operators=polynomial\n\n"
    "Now classify the following. Output exactly one line, no extra words.\n"
    "dialect: {dialect}\n"
    "expression: {expression}\n"
    "Output:"
)

REPAIR_TEMPLATE = (
    "Your previous output was incorrect. Rewrite it into exactly one line matching the expected format.\n"
    "Allowed family labels: Advection, Burgers, Heat, KleinGordon, Laplace, Wave\n"
    "Allowed operator labels: sin, cos, exp, tanh, polynomial\n"
    "Format Examples:\n"
    "family=Wave;operators=sin,cos\n"
    "family=Heat;operators=polynomial,exp\n"
    "No extra words.\n\n"
    "Text to fix: {previous_output}\n"
    "Output:"
)

## Cell 2 — Install & Imports

In [107]:
!pip install -q transformers torch

import json, re, random, time, datetime
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


## Cell 3 — Upload Dataset

In [108]:
FAMILY_CANON = {x.lower(): x for x in FAMILY_LABELS}
OP_SET       = set(OPERATOR_LABELS)

def parse_target(target_text):
    """Extract (family, operators) from raw target string."""
    fm = re.search(r"family\s*:\s*([^|]+)",    target_text, re.IGNORECASE)
    om = re.search(r"operators\s*:\s*([^|]+)", target_text, re.IGNORECASE)
    if not fm or not om:
        return None, None
    fam = FAMILY_CANON.get(fm.group(1).strip().lower())
    ops = sorted({x.strip().lower() for x in om.group(1).split(",") if x.strip() in OP_SET})
    return fam, ops if fam and ops else (None, None)

from google.colab import files
print("Upload your JSONL dataset file:")
uploaded = files.upload()
fname, raw_bytes = next(iter(uploaded.items()))

records = []
for line in raw_bytes.decode("utf-8").splitlines():
    line = line.strip()
    if line:
        rec = json.loads(line)
        rec["target_family"], rec["target_operators"] = parse_target(rec.get("target", ""))
        records.append(rec)

print(f"Loaded {len(records)} records from '{fname}'")

Upload your JSONL dataset file:


Saving dataset.jsonl to dataset (7).jsonl
Loaded 12000 records from 'dataset (7).jsonl'


## Cell 4 — Sample Indices per Dialect

In [118]:
rng = random.Random(CFG["seed"])
samples_by_dialect = {}

for dialect in CFG["dialects"]:
    eligible = [
        i for i, r in enumerate(records)
        if dialect in r.get("dialects", {})
        and r["target_family"] is not None
        and r["target_operators"] is not None
    ]
    n = min(CFG["n_target"], len(eligible))
    samples_by_dialect[dialect] = rng.sample(eligible, n)
    print(f"{dialect}: {n} sampled (eligible={len(eligible)})")

natural: 30 sampled (eligible=12000)
prefix: 30 sampled (eligible=12000)
postfix: 30 sampled (eligible=12000)
latex: 30 sampled (eligible=12000)


## Cell 5 — Load Model

In [119]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
model = AutoModelForSeq2SeqLM.from_pretrained(CFG["model_name"]).to(DEVICE)
model.eval()
print(f"Model '{CFG['model_name']}' loaded on {DEVICE}.")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model 'google/flan-t5-large' loaded on cuda.


## Cell 6 — Helpers: Generate & Parse

In [120]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        ids = model.generate(**inputs, do_sample=False, num_beams=1, repetition_penalty=1.5,
                             max_new_tokens=CFG["max_new_tokens"])
    return tokenizer.decode(ids[0], skip_special_tokens=True).strip()


def parse_output(raw):
    """Lenient parser: just looks for keyword presence in the raw string."""
    norm = raw.lower()

    # Find family
    fam = None
    for f in FAMILY_CANON:
        if f in norm:
            fam = FAMILY_CANON[f]
            break

    if not fam:
        return False, None, None, raw, "missing family"

    # Find operators
    ops = set()
    for op in OP_SET:
        if op in norm:
            ops.add(op)

    if not ops:
        return False, None, None, raw, "empty operators"

    return True, fam, sorted(list(ops)), raw, None

## Cell 7 — Debug Logger

In [121]:
RUN_ID = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
debug_events = []

def log(event_type, payload):
    debug_events.append({"event": event_type, "run_id": RUN_ID,
                         "ts": datetime.datetime.utcnow().isoformat(), **payload})

log("RUN_START", {
    "model_name":    CFG["model_name"],
    "seed":          CFG["seed"],
    "decode_config": {"do_sample": False, "num_beams": 1},
    "prompt_template": PROMPT_TEMPLATE,
})

/tmp/ipykernel_4343/2885393674.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_ID = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
/tmp/ipykernel_4343/2885393674.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.datetime.utcnow().isoformat(), **payload})


## Cell 8 — Main Evaluation Loop

In [122]:
global_retry_cnt  = 0
early_terminated  = False
termination_reason = None
final_rows         = []

for dialect in CFG["dialects"]:
    if early_terminated:
        break

    for sample_index, rec_idx in enumerate(samples_by_dialect[dialect]):
        if early_terminated:
            break

        rec        = records[rec_idx]
        expression = rec["dialects"][dialect]
        tgt_fam    = rec["target_family"]
        tgt_ops    = rec["target_operators"]

        log("SAMPLE_START", {"dialect": dialect, "sample_index": sample_index,
                             "source_record_index": rec_idx,
                             "input_expression": expression,
                             "target_family": tgt_fam, "target_operators": tgt_ops})

        raw_output     = ""
        final_status   = "NOT_EVEN_WRONG"
        parsed_family  = None
        parsed_ops     = []
        parse_error    = None
        attempts_used  = 0

        for attempt_id in range(1, CFG["max_attempts"] + 1):
            is_retry = attempt_id > 1
            if is_retry:
                global_retry_cnt += 1
                if global_retry_cnt > CFG["global_retry_limit"]:
                    early_terminated   = True
                    termination_reason = f"global_retry_cnt={global_retry_cnt} > {CFG['global_retry_limit']}"
                    log("RUN_TERMINATED_EARLY", {"termination_reason": termination_reason,
                                                  "global_retry_cnt": global_retry_cnt})
                    break

            prompt = (PROMPT_TEMPLATE.format(dialect=dialect, expression=expression)
                      if attempt_id == 1
                      else REPAIR_TEMPLATE.format(previous_output=raw_output))

            t0         = time.perf_counter()
            raw_output = generate(prompt)
            latency_ms = round((time.perf_counter() - t0) * 1000, 2)

            ok, fam, ops, norm, parse_error = parse_output(raw_output)
            attempts_used = attempt_id

            if ok:
                parsed_family = fam
                parsed_ops    = ops
                final_status  = "RIGHT" if (fam == tgt_fam and set(ops) == set(tgt_ops)) else "WRONG"
            else:
                final_status = "NOT_EVEN_WRONG"

            log("ATTEMPT_RESULT", {
                "dialect": dialect, "sample_index": sample_index,
                "attempt_id": attempt_id, "is_retry": is_retry,
                "global_retry_cnt_after_attempt": global_retry_cnt,
                "raw_model_output": raw_output, "normalized_output": norm,
                "parse_success": ok, "parsed_family": fam, "parsed_operators": ops,
                "parse_error": parse_error, "decision_after_attempt": final_status,
                "latency_ms": latency_ms,
            })

            if ok:
                break

        if early_terminated:
            break

        discarded = (final_status == "NOT_EVEN_WRONG")
        row = {
            "dialect": dialect, "sample_index": sample_index,
            "source_record_index": rec_idx, "input_expression": expression,
            "target_family": tgt_fam, "target_operators": tgt_ops,
            "raw_model_output": raw_output,
            "parsed_family": parsed_family, "parsed_operators": parsed_ops,
            "parse_error": parse_error, "attempts_used": attempts_used,
            "final_status": final_status, "discarded": discarded,
        }
        final_rows.append(row)
        log("SAMPLE_FINAL", row)

summary = {
    "run_id":            RUN_ID,
    "run_status":        "TERMINATED_EARLY" if early_terminated else "COMPLETED",
    "termination_reason": termination_reason,
    "processed_samples": len(final_rows),
    "global_retry_cnt":  global_retry_cnt,
    "discarded_total":   sum(1 for r in final_rows if r["discarded"]),
    "early_terminated":  early_terminated,
}
log("RUN_END", summary)
print(summary)

/tmp/ipykernel_4343/2885393674.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.datetime.utcnow().isoformat(), **payload})


{'run_id': '20260424T022027Z', 'run_status': 'COMPLETED', 'termination_reason': None, 'processed_samples': 120, 'global_retry_cnt': 50, 'discarded_total': 25, 'early_terminated': False}


## Cell 9 — Compute Metrics (only if run completed)

In [123]:
metrics_df = None

if not early_terminated:
    rows_list = []
    for dialect in CFG["dialects"]:
        dialect_rows = [r for r in final_rows if r["dialect"] == dialect]
        effective    = [r for r in dialect_rows if not r["discarded"]]
        n_target     = len(samples_by_dialect[dialect])
        n_eff        = len(effective)
        right        = sum(1 for r in effective if r["final_status"] == "RIGHT")
        wrong        = sum(1 for r in effective if r["final_status"] == "WRONG")
        rows_list.append({
            "dialect":           dialect,
            "n_target":          n_target,
            "discarded":         n_target - n_eff,
            "n_effective":       n_eff,
            "accuracy":          right / n_eff if n_eff else None,
            "f1_binary":         (2*right) / (2*right + wrong) if (2*right + wrong) else None,
            "parse_success_rate": n_eff / n_target if n_target else None,
        })
    metrics_df = pd.DataFrame(rows_list)
    display(metrics_df)
else:
    print("Run terminated early — metrics skipped.")

,dialect,n_target,discarded,n_effective,accuracy,f1_binary,parse_success_rate
0,natural,30,0,30,0.0,0.0,1.000000
1,prefix,30,0,30,0.0,0.0,1.000000
2,postfix,30,18,12,0.0,0.0,0.400000
3,latex,30,7,23,0.0,0.0,0.766667


## Cell 10 — Debug Output

In [124]:
print("=== RUN SUMMARY ===")
print(json.dumps(summary, indent=2))

print("\n=== PREDICTIONS (one row per sample) ===")
display(pd.DataFrame(final_rows))

print("\n=== ATTEMPT EVENTS (full trace) ===")
attempt_events = [e for e in debug_events if e["event"] == "ATTEMPT_RESULT"]
display(pd.DataFrame(attempt_events))

=== RUN SUMMARY ===
{
  "run_id": "20260424T022027Z",
  "run_status": "COMPLETED",
  "termination_reason": null,
  "processed_samples": 120,
  "global_retry_cnt": 50,
  "discarded_total": 25,
  "early_terminated": false
}

=== PREDICTIONS (one row per sample) ===


,dialect,sample_index,source_record_index,input_expression,target_family,target_operators,raw_model_output,parsed_family,parsed_operators,parse_error,attempts_used,final_status,discarded
0,natural,0,10476,In the y-z plane: 1.82 times second y-derivati...,Laplace,"[cos, exp, polynomial, sin]","family=Advection;operators=exp,tanh,exp,exp",Advection,"[exp, tanh]",None,1,WRONG,False
1,natural,1,1824,The curvature of u in time is proportional to ...,Wave,"[cos, exp, sin]","family=Advection;operators=exp,tanh,exp",Advection,"[exp, tanh]",None,1,WRONG,False
2,natural,2,409,The second time derivative of u equals 7.62 ti...,Wave,"[cos, exp, sin]","family=Advection;operators=exp,tanh,exp",Advection,"[exp, tanh]",None,1,WRONG,False
3,natural,3,4506,u propagates at speed 0.84 in x and y: second ...,Wave,"[cos, polynomial, sin]","family=Advection;operators=exp,tanh,exp,exp",Advection,"[exp, tanh]",None,1,WRONG,False
4,natural,4,4012,u moves without changing shape: u_t plus 1.13 ...,Advection,"[exp, polynomial]",family=Advection;operators=exp;family=KleinGor...,Advection,"[exp, tanh]",None,1,WRONG,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,latex,25,10738,0 = - 1.7136 \frac{\partial}{\partial z} u{\le...,Advection,"[exp, polynomial]","0 = -1.7136 fracpartialpartial z uleft(t,x,y,z...",None,[],missing family,3,NOT_EVEN_WRONG,True
116,latex,26,8179,- 1.2056 \frac{\partial^{2}}{\partial z^{2}} u...,Burgers,"[exp, polynomial, tanh]","Output: family=Advection;operators=exp,tanh,ex...",Advection,"[exp, tanh]",None,1,WRONG,False
117,latex,27,6482,- 1.4287 \frac{\partial^{2}}{\partial z^{2}} u...,Heat,"[exp, polynomial]","Output: family=Advection;operators=exp;print(""...",Advection,[exp],None,1,WRONG,False
118,latex,28,10532,- 4.68 \frac{\partial^{2}}{\partial z^{2}} u{\...,Wave,"[cos, exp, sin]","Output: family=Advection;operators=exp,tanh,ex...",Advection,"[exp, tanh]",None,1,WRONG,False



=== ATTEMPT EVENTS (full trace) ===


,event,run_id,ts,dialect,sample_index,attempt_id,is_retry,global_retry_cnt_after_attempt,raw_model_output,normalized_output,parse_success,parsed_family,parsed_operators,parse_error,decision_after_attempt,latency_ms
0,ATTEMPT_RESULT,20260424T022027Z,2026-04-24T02:20:29.337708,natural,0,1,False,0,"family=Advection;operators=exp,tanh,exp,exp","family=Advection;operators=exp,tanh,exp,exp",True,Advection,"[exp, tanh]",None,WRONG,1712.57
1,ATTEMPT_RESULT,20260424T022027Z,2026-04-24T02:20:30.176377,natural,1,1,False,0,"family=Advection;operators=exp,tanh,exp","family=Advection;operators=exp,tanh,exp",True,Advection,"[exp, tanh]",None,WRONG,837.89
2,ATTEMPT_RESULT,20260424T022027Z,2026-04-24T02:20:30.960294,natural,2,1,False,0,"family=Advection;operators=exp,tanh,exp","family=Advection;operators=exp,tanh,exp",True,Advection,"[exp, tanh]",None,WRONG,783.15
3,ATTEMPT_RESULT,20260424T022027Z,2026-04-24T02:20:31.959108,natural,3,1,False,0,"family=Advection;operators=exp,tanh,exp,exp","family=Advection;operators=exp,tanh,exp,exp",True,Advection,"[exp, tanh]",None,WRONG,998.15
4,ATTEMPT_RESULT,20260424T022027Z,2026-04-24T02:20:33.112723,natural,4,1,False,0,family=Advection;operators=exp;family=KleinGor...,family=Advection;operators=exp;family=KleinGor...,True,Advection,"[exp, tanh]",None,WRONG,1152.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,ATTEMPT_RESULT,20260424T022027Z,2026-04-24T02:26:09.640203,latex,25,3,True,50,"0 = -1.7136 fracpartialpartial z uleft(t,x,y,z...","0 = -1.7136 fracpartialpartial z uleft(t,x,y,z...",False,None,None,missing family,NOT_EVEN_WRONG,2041.11
166,ATTEMPT_RESULT,20260424T022027Z,2026-04-24T02:26:12.172244,latex,26,1,False,50,"Output: family=Advection;operators=exp,tanh,ex...","Output: family=Advection;operators=exp,tanh,ex...",True,Advection,"[exp, tanh]",None,WRONG,2531.95
167,ATTEMPT_RESULT,20260424T022027Z,2026-04-24T02:26:15.135673,latex,27,1,False,50,"Output: family=Advection;operators=exp;print(""...","Output: family=Advection;operators=exp;print(""...",True,Advection,[exp],None,WRONG,2963.30
168,ATTEMPT_RESULT,20260424T022027Z,2026-04-24T02:26:17.843845,latex,28,1,False,50,"Output: family=Advection;operators=exp,tanh,ex...","Output: family=Advection;operators=exp,tanh,ex...",True,Advection,"[exp, tanh]",None,WRONG,2708.03


In [125]:
# Quick diagnostic — run one raw generation and print the actual output
test_prompt = PROMPT_TEMPLATE.format(dialect="natural", expression="test expression")
test_out = generate(test_prompt)
print(repr(test_out))
print(type(test_out))

'family=Advection;operators=exp,tanh,exp'
<class 'str'>
